# Simulação Numérica e Animação de Escoamentos de Fluidos (Aula 01)
**Universidade Federal de Pelotas (UFPel)**  
**Programa de Pós-Graduação em Modelagem Matemática (PPGMMat)**  
**Disciplina:** Mecânica dos Fluidos (Prof. Dr. Régis S. de Quadros)

---

## Apresentação da Simulação Interativa
Este notebook é dedicado exclusivamente ao **Módulo de Animação e Vídeo Paramétrico de Escoamentos de Fluidos (Advecção Contínua - Modelo Euleriano)**.

Ele permite ao usuário alterar interativamente qualquer parâmetro físico da **Aula 01** ($U_\infty, \nu, \rho, L_c$) e escolher entre diferentes geometrias de escoamento:
1. **`'placa'`**: Camada limite laminar (Blasius) e desenvolvimento da espessura $\delta(x)$.
2. **`'cilindro'`**: Escoamento em torno de corpo rombudo e esteira oscilatória de vórtices de von Kármán.
3. **`'duto'`**: Região de entrada e desenvolvimento do perfil em canal 2D.
4. **`'poiseuille'`**: Escoamento totalmente desenvolvido em tubo circular (Laminar vs. Turbulento $1/7$).



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import map_coordinates
from IPython.display import HTML, Image

# Configuração de Estilo Visual dos Gráficos Matplotlib
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['lines.linewidth'] = 2.0
plt.style.use('seaborn-v0_8-whitegrid')

print("[OK] Ambiente configurado com sucesso! Bibliotecas numéricas e de animação prontas.")


## Módulo de Animação Interativa do Escoamento Contínuo de Fluido

Abaixo é implementada a função **`simular_video_fluido(...)`**, que calcula o campo de velocidades, realiza a advecção contínua do traçador e plota as linhas de corrente em tempo real.

### Parâmetros Configuráveis:
- **`geometria`**: `'placa'`, `'cilindro'`, `'duto'`, `'poiseuille'`
- **`U_inf`**: Velocidade da corrente livre ($0{,}1$ a $20{,}0 \text{ m/s}$)
- **`nu`**: Viscosidade cinemática ($10^{-6} \text{ m²/s}$ para água, $1.5 \times 10^{-5} \text{ m²/s}$ para ar)
- **`L_c`**: Dimensão característica (Raio do cilindro $R$, Comprimento da placa $L$, Altura do duto $H$)
- **`rho`**: Massa específica ($1{,}2 \text{ kg/m³}$ para ar, $1000 \text{ kg/m³}$ para água)
- **`num_frames`**: Número de quadros da animação (padrão: 30)
- **`salvar_gif`**: Salva automaticamente o arquivo de vídeo animado em `.gif`.



In [ ]:
def simular_video_fluido(geometria='placa', U_inf=2.0, nu=1.5e-5, rho=1.204, L_c=0.4, num_frames=30, salvar_gif=True, nome_gif='animacao_escoamento_aula01.gif'):
    # Recálculo dos Parâmetros Derivados da Aula 01
    Re = (U_inf * L_c) / nu
    Mach = U_inf / 340.0 # Velocidade do som no ar = 340 m/s
    drho_rho0 = 0.5 * Mach**2 * 100.0 # Variação relativa %
    
    if geometria == 'cilindro':
        regime = "Subcrítico (Separação Laminar)" if Re < 2e5 else "Supercrítico (Separação Turbulenta)"
        Cd_aprox = 1.1 if Re < 2e5 else 0.35
    elif geometria == 'placa':
        regime = "Laminar (Blasius)" if Re < 5e5 else "Turbulento"
        delta_L = (5.0 * L_c / np.sqrt(Re)) * 1000.0 if Re < 5e5 else (0.38 * L_c / (Re**0.2)) * 1000.0
        Cd_aprox = 1.328 / np.sqrt(Re) if Re < 5e5 else 0.074 / (Re**0.2)
    else:
        regime = "Laminar" if Re < 2300 else ("Transição" if Re <= 4000 else "Turbulento")
        Cd_aprox = 64.0 / Re if Re < 2300 else 0.316 / (Re**0.25)
        
    print("=========================================================================")
    print(f" SIMULAÇÃO CONTÍNUA DO FLUIDO (AULA 01) - GEOMETRIA: {geometria.upper()}")
    print("=========================================================================")
    print(f"  - Velocidade U_inf: {U_inf:.2f} m/s | Viscosidade nu: {nu:.2e} m²/s | rho: {rho:.1f} kg/m³")
    print(f"  - Dimensão L_c: {L_c:.3f} m | Número de Reynolds Re: {Re:.1f} ({regime})")
    print(f"  - Número de Mach: {Mach:.4f} (Compressibilidade drho/rho0: {drho_rho0:.2f}%)")
    print(f"  - Coeficiente de Arrasto Estimado Cd: {Cd_aprox:.4f}")
    if geometria == 'placa':
        print(f"  - Espessura da Camada Limite no final delta(L): {delta_L:.2f} mm")
    print("=========================================================================")

    # Configuração da Grade Numérica 2D (Euleriana para Advecção Contínua do Fluido)
    Lx, Ly = 4.0, 2.0
    nx_g, ny_g = 160, 90
    xg = np.linspace(0, Lx, nx_g)
    yg = np.linspace(0, Ly, ny_g)
    Xg, Yg = np.meshgrid(xg, yg)
    dx = xg[1] - xg[0]
    dy = yg[1] - yg[0]
    
    # Campo de Velocidades Base Conforme Geometria
    if geometria == 'cilindro':
        cx, cy = 1.5, 1.0
        R = L_c
        r_dist = np.hypot(Xg - cx, Yg - cy)
        r_dist_clamped = np.maximum(r_dist, R)
        theta_angle = np.arctan2(Yg - cy, Xg - cx)
        
        u_x_base = U_inf * (1.0 - (R**2 / r_dist_clamped**2) * np.cos(2.0 * theta_angle))
        u_y_base = -U_inf * (R**2 / r_dist_clamped**2) * np.sin(2.0 * theta_angle)
        
        mask_obs = r_dist < R
        u_x_base[mask_obs] = 0.0
        u_y_base[mask_obs] = 0.0
        
    elif geometria == 'placa':
        u_x_base = np.ones_like(Xg) * U_inf
        u_y_base = np.zeros_like(Yg)
        mask_obs = (Yg <= 0.01) & (Xg >= 0.5) & (Xg <= 3.5)
        for i in range(ny_g):
            for j in range(nx_g):
                x_pos = Xg[i, j]
                y_pos = Yg[i, j]
                if x_pos >= 0.5:
                    Re_loc = max(U_inf * (x_pos - 0.5) / nu, 1.0)
                    delta_loc = 5.0 * (x_pos - 0.5) / np.sqrt(Re_loc)
                    if y_pos <= delta_loc:
                        eta_rel = y_pos / max(delta_loc, 1e-6)
                        u_x_base[i, j] = U_inf * (1.5 * eta_rel - 0.5 * eta_rel**3)
                        u_y_base[i, j] = U_inf * 0.05 * (1.0 - eta_rel)

    elif geometria == 'duto':
        u_x_base = np.zeros_like(Xg)
        u_y_base = np.zeros_like(Yg)
        H_duto = 1.4
        y_mid = 1.0
        mask_obs = (Yg < 0.3) | (Yg > 1.7)
        for i in range(ny_g):
            y_pos = Yg[i, 0]
            if 0.3 <= y_pos <= 1.7:
                r_rel = (y_pos - y_mid) / (H_duto / 2.0)
                u_x_base[i, :] = 1.5 * U_inf * (1.0 - r_rel**2)
                
    else: # Poiseuille em Tubo
        u_x_base = np.zeros_like(Xg)
        u_y_base = np.zeros_like(Yg)
        mask_obs = (Yg < 0.2) | (Yg > 1.8)
        for i in range(ny_g):
            r_norm = (Yg[i, 0] - 1.0) / 0.8
            if np.abs(r_norm) <= 1.0:
                if Re < 2300:
                    u_x_base[i, :] = 2.0 * U_inf * (1.0 - r_norm**2)
                else:
                    u_x_base[i, :] = (60.0 / 49.0) * U_inf * (1.0 - np.abs(r_norm))**(1.0/7.0)

    # Inicialização do Campo Escalar Contínuo do Fluido (Tracer Dye / Tiras de Corrente)
    C = 0.5 + 0.5 * np.sin(10.0 * np.pi * Yg / Ly)
    
    fig, ax = plt.subplots(figsize=(11, 5.5))
    
    # Renderização da Advecção do Campo Contínuo do Fluido (Continuidade Euleriana)
    im = ax.imshow(C, origin='lower', extent=[0, Lx, 0, Ly], cmap='viridis', vmin=0, vmax=1, aspect='auto', zorder=1)
    cbar = fig.colorbar(im, ax=ax, label=r'Concentração / Traçador Contínuo do Fluido $C(x,y,t)$')
    
    # Overlay das Linhas de Corrente (Streamlines)
    ax.streamplot(xg, yg, u_x_base, u_y_base, color='white', linewidth=0.6, density=0.7, arrowsize=0.7, zorder=3)
    
    # Desenho dos Obstáculos e Geometrias
    if geometria == 'cilindro':
        circle = plt.Circle((cx, cy), R, color='crimson', zorder=10, label='Cilindro Sólido')
        ax.add_patch(circle)
    elif geometria == 'placa':
        ax.plot([0.5, 3.5], [0, 0], color='crimson', lw=6, zorder=10, label='Placa Plana')
        # Curva da Espessura da Camada Limite delta(x)
        x_bl = np.linspace(0.5, 3.5, 100)
        Re_x_bl = np.maximum(U_inf * (x_bl - 0.5) / nu, 1.0)
        delta_bl = 5.0 * (x_bl - 0.5) / np.sqrt(Re_x_bl)
        ax.plot(x_bl, delta_bl, 'r--', lw=1.8, label=r'Fronteira $\delta(x)$')
    elif geometria == 'duto':
        ax.axhline(0.3, color='black', lw=5, zorder=10, label='Parede do Duto')
        ax.axhline(1.7, color='black', lw=5, zorder=10)
    elif geometria == 'poiseuille':
        ax.axhline(0.2, color='black', lw=5, zorder=10, label='Parede do Tubo')
        ax.axhline(1.8, color='black', lw=5, zorder=10)

    title_text = f"Escoamento Contínuo de Fluido ({geometria.upper()}) | U_inf={U_inf:.1f} m/s | Re={Re:.0f} ({regime})"
    ax.set_title(title_text, fontsize=11, fontweight='bold')
    ax.set_xlabel(r'Posição Longitudinal $x$ [m]')
    ax.set_ylabel(r'Posição Transversal $y$ [m]')
    ax.set_xlim(0, Lx)
    ax.set_ylim(0, Ly)
    
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='upper right')

    dt = 0.04
    t_curr = 0.0

    def update(frame):
        nonlocal C, t_curr
        t_curr += dt
        
        # Campo de Velocidades Instantâneo (inclui esteira oscilatória para cilindro)
        u_x = u_x_base.copy()
        u_y = u_y_base.copy()
        
        if geometria == 'cilindro' and Re > 40:
            # Oscilação física da esteira de von Kármán no campo de fluido
            wake_mask = (Xg > cx) & (np.abs(Yg - cy) < R * 1.5)
            strouhal = 0.2
            freq = strouhal * U_inf / (2 * R)
            v_wake = 0.3 * U_inf * np.sin(2.0 * np.pi * freq * t_curr - 4.0 * (Xg - cx)) * np.exp(-((Yg - cy) / R)**2)
            u_y[wake_mask] += v_wake[wake_mask]

        # Advecção Semi-Lagrangiana do Campo Contínuo do Fluido
        X_back = Xg - u_x * dt
        Y_back = Yg - u_y * dt
        
        ix = X_back / dx
        iy = Y_back / dy
        coords = np.array([iy.ravel(), ix.ravel()])
        
        C_next = map_coordinates(C, coords, order=1, mode='nearest').reshape((ny_g, nx_g))
        
        # Condição de Contorno na Entrada (Injeção Contínua de Tiras do Fluido)
        inlet_mask = X_back < 0
        C_next[inlet_mask] = 0.5 + 0.5 * np.sin(10.0 * np.pi * Yg[inlet_mask] / Ly)
        C_next[mask_obs] = 0.0
        
        C = C_next
        im.set_data(C)
        return im,

    anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=60, blit=True)
    
    if salvar_gif:
        try:
            anim.save(nome_gif, writer='pillow', fps=15)
            print(f"[SUCCESS] Animação exportada e salva com sucesso em: {nome_gif}")
        except Exception as e:
            print(f"[AVISO] Não foi possível salvar GIF: {e}")
            
    plt.tight_layout()
    plt.show()
    return anim

# --- Exemplo 1: Simulação para Placa Plana ---
print("Executando simulação para PLACA PLANA...")
anim_placa = simular_video_fluido(geometria='placa', U_inf=4.0, nu=1.5e-5, L_c=0.4, num_frames=30, salvar_gif=True, nome_gif='animacao_escoamento_placa.gif')

# --- Exemplo 2: Simulação para Cilindro ---
print("\nExecutando simulação para CILINDRO...")
anim_cilindro = simular_video_fluido(geometria='cilindro', U_inf=2.0, nu=1.5e-5, L_c=0.4, num_frames=30, salvar_gif=True, nome_gif='animacao_escoamento_aula01.gif')
